In [1]:
%pip install sickle 
%pip install sentence-transformers chromadb

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.1 MB 869.0 kB/s eta 0:00:05
   ------- -------------------------------- 0.8/4.1 MB 887.4 kB/s eta 0:00:04
   ------- -------------------------------- 0.8/4.1 MB 887.4 kB/s eta 0:00:04
   ---------- ----------------------------- 1.0/4.1 MB 683.2 kB/s eta 0:00:05
   ---------- ----------------------------- 1.0/4.1 MB 683.2 kB/s eta 0:00:05
   ---------- ----------------------------- 1.0/4.1 MB 683.2 kB/s eta 0:00:05
   ---------- --------------------------

In [2]:
import json
from sickle import Sickle
import chromadb
from sentence_transformers import SentenceTransformer

C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
sickle = Sickle('https://eprints.undip.ac.id/cgi/oai2')

print("Memulai proses harvesting dan filtering data ber-abstrak...")

try:
    records = sickle.ListRecords(metadataPrefix='oai_dc')

    harvested_data = []
    target_count = 1000  # Target data valid yang ingin dikumpulkan
    count = 0
    scanned = 0

    for record in records:
        scanned += 1
        
        # PENGAMAN UTAMA: Tangkap error jika record kosong atau tidak punya metadata (deleted record)
        try:
            metadata = record.metadata
        except AttributeError:
            continue  # Lewati record yang rusak/tidak punya metadata

        # Ekstraksi field dengan aman
        title = metadata.get('title', [''])[0]
        creators = metadata.get('creator', [])
        description = metadata.get('description', [''])[0] # Ini abstrak
        identifiers = metadata.get('identifier', [])
        dates = metadata.get('date', [])
        
        # Ambil tanggal pertama jika tersedia
        publication_date = dates[0] if dates else 'Tanggal tidak diketahui'
        
        # Cari URL dokumen asli
        url = next((i for i in identifiers if 'http' in i), '')

        # FILTER TAMBAHAN: Singkirkan dokumen administratif (lembar penilaian/peer review)
        is_administrative = "penilaian sejawat" in title.lower() or "peer_review" in url.lower()

        # Syarat valid: Deskripsi ada, panjang > 50 karakter, dan bukan dokumen administratif
        if description and len(description.strip()) > 50 and not is_administrative:
            item = {
                "title": title,
                "creators": creators,
                "description": description,
                "url": url,
                "date": publication_date
            }
            
            harvested_data.append(item)
            count += 1
            print(f"[Valid #{count}] (Scan ke-{scanned}) Dapatkan: {title[:50]}...")

            # Berhenti jika target sudah tercapai
            if count >= target_count:
                break

    # Simpan hasil panen yang sudah bersih ke file JSON
    with open('eprints_undip_clean.json', 'w', encoding='utf-8') as f:
        json.dump(harvested_data, f, ensure_ascii=False, indent=4)

    print(f"\nSelesai! Berhasil mengamankan {len(harvested_data)} data valid ke 'eprints_undip_clean.json'.")

except Exception as e:
    print(f"Terjadi kendala umum saat koneksi/harvesting: {e}")

Memulai proses harvesting dan filtering data ber-abstrak...
[Valid #1] (Scan ke-189) Dapatkan: Determinants of the Stock Price Volatility
in the...
[Valid #2] (Scan ke-239) Dapatkan: Community Behavior Towards Filariasis Mass Drug Ad...
[Valid #3] (Scan ke-355) Dapatkan: Identification of Plasmodium Falciparum Phase in R...
[Valid #4] (Scan ke-361) Dapatkan: Identification of Plasmodium Falciparum Developmen...
[Valid #5] (Scan ke-370) Dapatkan: Volume Target Delineation for Brain Tumor in Mri I...
[Valid #6] (Scan ke-371) Dapatkan: Identifying the Developmental Phase of Plasmodium ...
[Valid #7] (Scan ke-431) Dapatkan: PERUMUSAN TUJUAN  LAPORAN KEUANGAN ISLAMI: PENDEKA...
[Valid #8] (Scan ke-433) Dapatkan: ASUMSI DASAR ILMU EKONOMI   ISLAM DAN   HALL MARK ...
[Valid #9] (Scan ke-507) Dapatkan: POTENSI UMKM KREATIF MENUJU PEMBENTUKAN DAN PENGUA...
[Valid #10] (Scan ke-547) Dapatkan: The Mission Of Basic Education That Is Overlooked ...
[Valid #11] (Scan ke-555) Dapatkan: Designing Hydr

In [4]:
print("1. Memuat model AI Embedding (multilingual)...")
# Model ini sangat bagus untuk bahasa Indonesia & Inggris secara bersamaan
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print("2. Menghubungkan ke ChromaDB (database vektor lokal)...")
client = chromadb.PersistentClient(path="./eprints_vector_db")

# Hapus koleksi lama jika ingin memperbarui dari awal
try:
    client.delete_collection(name="undip_research_search")
except Exception:
    pass

collection = client.get_or_create_collection(
    name="undip_research_search",
    metadata={"hnsw:space": "cosine"} # Mengubah metrik jarak menjadi Cosine
)

print("3. Membaca data dari file JSON hasil panen...")
with open('eprints_undip_clean.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

documents = []
metadatas = []
ids = []

for idx, item in enumerate(data):
    # Gabungkan Judul dan Deskripsi/Abstrak agar konteks pencariannya makin kaya
    text_to_embed = f"{item['title']}. {item['description']}"
    
    doc_id = f"doc_{idx}"
    documents.append(text_to_embed)
    
    # Ambil tanggal dengan aman (mengantisipasi list atau string kosong)
    raw_date = item.get("date", "")
    date_str = raw_date[0] if isinstance(raw_date, list) and raw_date else str(raw_date)

    metadatas.append({
        "title": item["title"],
        "creators": ", ".join(item["creators"]) if isinstance(item["creators"], list) else str(item["creators"]),
        "url": item["url"],
        "description": item["description"],
        "date": date_str
    })
    ids.append(doc_id)

print("4. Melakukan embedding massal secara kilat (Batch Encoding)...")
# Menggunakan batch_size=64 dengan progress bar agar proses 10.000 data cepat dan terpantau
embeddings = model.encode(documents, batch_size=64, show_progress_bar=True).tolist()

print("5. Memasukkan vektor dan metadata ke ChromaDB secara bertahap (Batch 1000)...")
batch_size = 1000  # Batas aman di bawah limit maksimal ChromaDB (5461)
total_data = len(ids)

for i in range(0, total_data, batch_size):
    end_idx = min(i + batch_size, total_data)
    
    # Masukkan per batch ke database
    collection.add(
        ids=ids[i:end_idx],
        embeddings=embeddings[i:end_idx],
        metadatas=metadatas[i:end_idx],
        documents=documents[i:end_idx]
    )
    print(f"Progress: Berhasil menyimpan data ke-{i} sampai {end_idx}...")

print(f"\nSukses! Seluruh {total_data} dokumen berhasil di-vektorisasi dan disimpan ke Vector Database.")

1. Memuat model AI Embedding (multilingual)...


C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\

2. Menghubungkan ke ChromaDB (database vektor lokal)...
3. Membaca data dari file JSON hasil panen...
4. Melakukan embedding massal secara kilat (Batch Encoding)...


Batches: 100%|██████████| 16/16 [00:53<00:00,  3.31s/it]


5. Memasukkan vektor dan metadata ke ChromaDB secara bertahap (Batch 1000)...
Progress: Berhasil menyimpan data ke-0 sampai 1000...

Sukses! Seluruh 1000 dokumen berhasil di-vektorisasi dan disimpan ke Vector Database.


In [5]:
# Load model & database yang sudah dibuat
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
client = chromadb.PersistentClient(path="./eprints_vector_db")
collection = client.get_collection(name="undip_research_search")

# Simulasi user mengetik query pencarian dengan bahasa bebas / konsep (bukan keyword pas)
query = "penelitian tentang kemiskinan"
MAX_DISTANCE_THRESHOLD = 0.5
print(f"Query User: '{query}'\n" + "="*50)

# Ubah query user jadi vektor
query_vector = model.encode(query).tolist()

# Cari 3 dokumen yang paling mirip maknanya di database
results = collection.query(
    query_embeddings=[query_vector],
    n_results=100,
    include=['metadatas', 'distances']
)

json_results = []

for i, (metadata, distance) in enumerate(zip(results['metadatas'][0], results['distances'][0])):
    # FILTER THRESHOLD: Lewati dokumen jika jaraknya melebihi batas maksimal
    if distance > MAX_DISTANCE_THRESHOLD:
        continue # Skip dokumen yang tidak memenuhi syarat relevansi
    
    # Di ChromaDB (dengan Cosine distance), nilai distance yang SEMAKIN KECIL (mendekati 0.0) 
    # artinya SEMAKIN MIRIP/RELEVAN.
    print(f"Peringkat #{i+1} (Skor Jarak: {distance:.4f})")
    print(f"Judul   : {metadata['title']}")
    print(f"Penulis : {metadata['creators']}")
    print(f"url: {metadata['url']}")
    print(f"date: {metadata['date']}")
    print("-" * 60)
    
    json_results.append({
        "peringkat": i+1,
        "skor": distance,
        "judul": metadata['title'],
        "penulis": metadata['creators'],
        "deskripsi": metadata['description'],
        "url": metadata['url'],
        "date": metadata['date'] 
    })

print(json_results)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1019.10it/s]


Query User: 'penelitian tentang kemiskinan'
Peringkat #1 (Skor Jarak: 0.3314)
Judul   : Klasifikasi Status Kemiskinan Rumah Tangga dengan Metode Support Vector Machines (SVM) dan Classification and Regression Trees (CART) Menggunakan GUI R 
(Studi Kasus di Kabupaten Wonosobo Tahun 2018)
Penulis : LUTFIA, NUZULA
url: https://eprints2.undip.ac.id/id/eprint/4743/1/1.%20COVER%201.pdf
date: 2020-09-21
------------------------------------------------------------
Peringkat #2 (Skor Jarak: 0.4347)
Judul   : HUBUNGAN TINGKAT KONSUMSI ENERGI,PROTEIN DAN LEMAK DENGAN STATUS GIZI	(STUDI KASUS PADA NARAPIDANA WANITA DI LEMBAGA PEMASYARAKATAN WANITA KELAS II A KOTA SEMARANG TAHUN 2021)
Penulis : AMALINA, DEVIA DWI
url: https://eprints2.undip.ac.id/id/eprint/4687/1/REPO%20DEVIA.pdf
date: 2021
------------------------------------------------------------
Peringkat #3 (Skor Jarak: 0.4641)
Judul   : HUBUNGAN PENGETAHUAN, PRAKTIK PEMBERIAN MAKAN BAYI DAN ANAK (PMBA) DAN PENGGUNAAN GARAM BERYODIUM DENGAN K